# PeoplePulse: Visualisation Features

This notebook prepares the visual evidence used by the PeoplePulse attrition dashboard.

It follows the same data and model path as the Streamlit app:

- cleaned HR employee data from `datasets/cleaned-data/HR-Employee-Attrition-cleaned.csv`
- historical attrition measured with `AttritionBinary`
- predicted attrition risk from the persisted pipeline in `models/employee_attrition_pipeline.joblib`
- actionable risk bands: Low (<30%), Medium (30-59%), and High (60%+)

The charts focus on workforce hotspots and interpretable signals for retention planning, not individual judgement.

In [1]:
from pathlib import Path
import sys

import joblib
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import display

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "datasets").exists():
    PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from ml_model import feature_columns, risk_band

DATA_PATH = PROJECT_ROOT / "datasets" / "cleaned-data" / "HR-Employee-Attrition-cleaned.csv"
MODEL_PATH = PROJECT_ROOT / "models" / "employee_attrition_pipeline.joblib"

PLOT_TEMPLATE = "plotly_white"
ACCENT = "#1565c0"
TEAL = "#00897b"
GOLD = "#f59e0b"
RED = "#d64545"

In [2]:
data = pd.read_csv(DATA_PATH)
model = joblib.load(MODEL_PATH)

features = feature_columns(data)
data["Risk probability"] = model.predict_proba(data[features])[:, 1]
data["Risk band"] = data["Risk probability"].map(risk_band)

data.shape, data["AttritionBinary"].mean()

((1470, 39), 0.16122448979591836)

## Visualisation 1: Workforce Composition

This treemap shows how employees are distributed across departments and job roles.

In [5]:
composition = (
    data.groupby(["Department", "JobRole"], as_index=False)
    .size()
    .rename(columns={"size": "Employees"})
    .sort_values("Employees", ascending=False)
)

fig = px.treemap(
    composition,
    path=["Department", "JobRole"],
    values="Employees",
    color="Employees",
    color_continuous_scale=["#d8f0ec", TEAL, "#102a43"],
    title="Workforce composition by department and job role",
    template=PLOT_TEMPLATE,
)
fig.update_layout(height=520, margin=dict(t=70, l=10, r=10, b=10))
fig.show()

### Findings

- The treemap compares workforce size across departments and roles.
- Larger rectangles represent roles with more employees.
- Use this view to add workforce context before comparing attrition rates.

## Visualisation 2: Historical Attrition by Job Role

This chart compares the historical attrition rate across job roles.

In [6]:
role_rates = (
    data.groupby("JobRole", as_index=False)
    .agg(
        Employees=("EmployeeNumber", "size"),
        AttritionRate=("AttritionBinary", "mean"),
    )
    .assign(AttritionRate=lambda frame: frame["AttritionRate"] * 100)
    .sort_values("AttritionRate")
)

fig = px.bar(
    role_rates,
    x="AttritionRate",
    y="JobRole",
    orientation="h",
    text=role_rates["AttritionRate"].map(lambda value: f"{value:.1f}%"),
    color="AttritionRate",
    color_continuous_scale=["#8bd3c7", GOLD, RED],
    hover_data={"Employees": True, "AttritionRate": ":.1f"},
    labels={"AttritionRate": "Historical attrition rate (%)", "JobRole": ""},
    title="Historical attrition rate by job role",
    template=PLOT_TEMPLATE,
)
fig.update_layout(
    coloraxis_showscale=False,
    height=520,
    margin=dict(t=70, l=10, r=10, b=10),
)
fig.show()

### Findings

- Attrition varies substantially by job role.
- Sales Representative has the highest historical attrition rate in this dataset.
- Research Director and Manager have comparatively lower historical attrition rates.
- Role-level differences should guide investigation, not determine decisions about individuals.

## Visualisation 3: Overtime and Department Attrition

This grouped bar chart compares attrition for employees who work overtime and those who do not, within each department.

In [7]:
signal_data = (
    data.groupby(["Department", "OverTime"], as_index=False)
    .agg(
        Employees=("EmployeeNumber", "size"),
        AttritionRate=("AttritionBinary", "mean"),
    )
)
signal_data["AttritionRate"] *= 100

fig = px.bar(
    signal_data,
    x="Department",
    y="AttritionRate",
    color="OverTime",
    barmode="group",
    text=signal_data["AttritionRate"].map(lambda value: f"{value:.1f}%"),
    color_discrete_map={"No": TEAL, "Yes": RED},
    hover_data={"Employees": True, "AttritionRate": ":.1f"},
    labels={
        "AttritionRate": "Historical attrition rate (%)",
        "OverTime": "Works overtime",
    },
    title="Overtime is an important retention signal across departments",
    template=PLOT_TEMPLATE,
)
fig.update_layout(height=460, margin=dict(t=70, l=10, r=10, b=10))
fig.show()

### Findings

- Overtime employees show higher historical attrition in every department.
- The overtime gap is especially visible in Sales and Research & Development.
- Workload, staffing, flexibility, and manager support are useful areas for follow-up investigation.

## Visualisation 4: Attrition by Monthly Income Quartile

This chart compares historical attrition across four groups formed from monthly income.

In [8]:
income_view = (
    data.assign(
        IncomeQuartile=pd.qcut(
            data["MonthlyIncome"],
            q=4,
            duplicates="drop",
        )
    )
    .groupby("IncomeQuartile", observed=True, as_index=False)
    .agg(
        Employees=("EmployeeNumber", "size"),
        AttritionRate=("AttritionBinary", "mean"),
    )
)
income_view["IncomeQuartile"] = income_view["IncomeQuartile"].astype(str)
income_view["AttritionRate"] *= 100

fig = px.bar(
    income_view,
    x="IncomeQuartile",
    y="AttritionRate",
    text=income_view["AttritionRate"].map(lambda value: f"{value:.1f}%"),
    color="AttritionRate",
    color_continuous_scale=["#8bd3c7", GOLD, RED],
    labels={
        "AttritionRate": "Historical attrition rate (%)",
        "IncomeQuartile": "Monthly income quartile",
    },
    title="Attrition across monthly income quartiles",
    template=PLOT_TEMPLATE,
)
fig.update_layout(
    coloraxis_showscale=False,
    height=440,
    margin=dict(t=70, l=10, r=10, b=10),
)
fig.show()

### Findings

- The lowest monthly-income quartile has the highest historical attrition rate.
- Attrition declines across the higher income quartiles in this dataset.
- This pattern supports investigating pay fairness, reward transparency, and career progression.

## Visualisation 5: Predicted Attrition Risk Distribution

This histogram shows how the persisted model distributes employees across predicted attrition probabilities and risk bands.

In [9]:
fig = px.histogram(
    data,
    x="Risk probability",
    color="Risk band",
    nbins=25,
    category_orders={"Risk band": ["Low", "Medium", "High"]},
    color_discrete_map={"Low": TEAL, "Medium": GOLD, "High": RED},
    title="Predicted attrition risk distribution",
    labels={
        "Risk probability": "Predicted attrition probability",
        "count": "Employees",
    },
    template=PLOT_TEMPLATE,
)
fig.update_xaxes(tickformat=".0%")
fig.update_layout(
    barmode="stack",
    height=460,
    margin=dict(t=70, l=10, r=10, b=10),
)
fig.show()

### Findings

- The model assigns each employee a predicted attrition probability.
- Risk bands make the continuous probabilities easier to use for triage: Low, Medium, and High.
- The model is decision support and should be combined with human context before action.

## Visualisation 6: High-Risk Profiles by Job Role

This chart identifies job roles with the largest share of profiles in the model's High risk band.

In [10]:
risk_by_role = (
    data.groupby("JobRole", as_index=False)
    .agg(
        Employees=("EmployeeNumber", "size"),
        HighRiskProfiles=(
            "Risk band",
            lambda values: (values == "High").sum(),
        ),
        AveragePredictedRisk=("Risk probability", "mean"),
    )
)
risk_by_role["HighRiskShare"] = (
    risk_by_role["HighRiskProfiles"] / risk_by_role["Employees"]
)
risk_by_role = risk_by_role.sort_values("HighRiskShare", ascending=False)

fig = px.bar(
    risk_by_role,
    x="HighRiskShare",
    y="JobRole",
    orientation="h",
    color="AveragePredictedRisk",
    text=risk_by_role["HighRiskShare"].map(lambda value: f"{value:.1%}"),
    color_continuous_scale=["#8bd3c7", GOLD, RED],
    hover_data={
        "Employees": True,
        "HighRiskProfiles": True,
        "AveragePredictedRisk": ":.1%",
    },
    labels={
        "HighRiskShare": "Share of profiles in High risk band",
        "JobRole": "",
    },
    title="Where high-risk profiles are concentrated",
    template=PLOT_TEMPLATE,
)
fig.update_xaxes(tickformat=".0%")
fig.update_layout(height=520, margin=dict(t=70, l=10, r=10, b=10))
fig.show()

### Findings

- High-risk profiles are concentrated unevenly across job roles.
- Sales Representative has the largest High-risk share in this model output.
- This view helps prioritise team-level retention conversations and further investigation.

## Conclusion

The visual analysis shows that employee attrition is concentrated around several connected workplace signals. Sales Representative and Laboratory Technician roles have comparatively high historical attrition, while employees who work overtime show higher attrition across every department. The lowest monthly-income group also records the highest attrition rate, suggesting that workload, pay fairness, and career progression should be investigated together rather than in isolation.

The machine-learning model identified 328 high-risk profiles from the 1,470 employees analysed. These risk scores can help PeoplePulse prioritise supportive retention conversations and team-level investigation. They should not be treated as definitive judgements about individuals. HR teams should validate the model signals with employee feedback, manager context, workload information, and fair employment practices before taking action.

Overall, the analysis provides an evidence-led starting point for retention planning: focus first on exposed roles and overtime-heavy teams, review reward and progression pathways, and monitor whether targeted interventions reduce attrition over time.